# 01. Synthetic SLC stack with known scatterers

This notebook constructs the in-memory complex SLC stack model that the interferometric stages consume. It plants a small set of point scatterers at known azimuth, range and physical height, encodes the topographic phase per track through the interferometric wavenumber kz, and confirms by eye that the amplitude images and the per-track phase ramps match the planted geometry. No file IO is performed: the real pipeline reads SLCs from disk via PyRat, which is unavailable here, so this synthetic stack stands in for those arrays.

**Modules exercised:** configuration.processing_config (TomogramConfiguration), the data contract consumed by pipelines.processing_pipeline.interferogram

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Stack parameters

A SLC stack has shape `(n_tracks, n_azimuth, n_range)` and complex dtype. Each track sees the same scene from a slightly different baseline; the baseline is summarised by the interferometric wavenumber `kz`, which maps physical height to interferometric phase via `phase = kz * height`.

In [ ]:
n_tracks  = 6
n_azimuth = 64
n_range   = 96

kz = np.linspace(0.0, 0.30, n_tracks)
print('kz (rad/m) per track:', np.round(kz, 4))

scatterers = [
    (16, 24, 2.5, 1.0, 5.0),
    (32, 60, 3.0, 0.8, 30.0),
    (48, 40, 2.0, 0.6, 55.0),
]
print('planted scatterers (az, rg, sigma, amp, height):')
for s in scatterers:
    print('  ', s)

## Stack synthesis helper

The helper plants Gaussian reflectivity blobs, assigns each its physical height, and applies the per-track topographic phase plus complex speckle noise.

In [ ]:
def synthetic_slc_stack(n_tracks, n_azimuth, n_range, kz, scatterers, rng, noise_sigma=0.05):
    azimuth_axis = np.arange(n_azimuth)
    range_axis   = np.arange(n_range)
    az_grid, rg_grid = np.meshgrid(azimuth_axis, range_axis, indexing="ij")

    reflectivity = np.zeros((n_azimuth, n_range), dtype=np.float32)
    heights      = np.zeros((n_azimuth, n_range), dtype=np.float32)

    for az_center, rg_center, sigma, amplitude, height in scatterers:
        blob          = amplitude * np.exp(-(((az_grid - az_center) ** 2 + (rg_grid - rg_center) ** 2) / (2.0 * sigma ** 2)))
        reflectivity += blob.astype(np.float32)
        heights       = np.where(blob > reflectivity * 0.5, height, heights).astype(np.float32)

    stack = np.empty((n_tracks, n_azimuth, n_range), dtype=np.complex64)
    for track_index in range(n_tracks):
        topographic_phase  = kz[track_index] * heights
        amplitude_field    = reflectivity
        complex_field      = amplitude_field * np.exp(1.0j * topographic_phase)
        speckle            = (rng.standard_normal(complex_field.shape) + 1.0j * rng.standard_normal(complex_field.shape))
        complex_field     += noise_sigma * speckle.astype(np.complex64)
        stack[track_index] = complex_field.astype(np.complex64)

    return stack, reflectivity, heights


In [ ]:
stack, reflectivity, heights = synthetic_slc_stack(
    n_tracks   = n_tracks,
    n_azimuth  = n_azimuth,
    n_range    = n_range,
    kz         = kz,
    scatterers = scatterers,
    rng        = RNG,
)
print('stack shape :', stack.shape, stack.dtype)
print('reflectivity:', reflectivity.shape)
print('heights     :', heights.shape, 'unique:', np.unique(np.round(heights)))

## Amplitude is shared across tracks

All tracks observe the same scene, so their amplitude images should look the same up to speckle. The planted scatterers should appear at their nominal azimuth and range positions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, t in zip(axes, [0, n_tracks // 2, n_tracks - 1]):
    im = ax.imshow(np.abs(stack[t]), aspect='auto', origin='lower')
    ax.set_title(f'|SLC| track {t}')
    ax.set_xlabel('range')
    ax.set_ylabel('azimuth')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
fig.tight_layout()
plt.show()

## Phase encodes height through kz

For a given scatterer the phase across tracks grows linearly with `kz` at a rate set by its height. The highest scatterer should show the steepest phase progression.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for (az, rg, _sigma, _amp, height) in scatterers:
    phase_track = np.angle(stack[:, az, rg])
    ax.plot(kz, np.unwrap(phase_track), marker='o', label=f'h={height:.0f} m')
ax.set_xlabel('kz (rad/m)')
ax.set_ylabel('unwrapped phase (rad)')
ax.set_title('Phase vs kz at each planted scatterer')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The three amplitude panels should be near-identical, each showing the three planted blobs at their nominal positions. The phase-versus-kz lines should be straight, with slope increasing with scatterer height, confirming the stack encodes `phase = kz * height` as intended.